<a href="https://colab.research.google.com/github/swaggincoffee-bit/Tesi/blob/main/00_preparazione_dati/01_Preparazione_dati_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 1 — Mount Google Drive
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")

import os

BASE      = "/content/drive/MyDrive/Contatori/CSV"
PATH_SAP  = os.path.join(BASE, "Parco Contatori_G4-G6_20251231_SAP.csv")
PATH_BEAM = os.path.join(BASE, "TABELLA_CONTATORE_BEAM.csv")
PATH_LETT = os.path.join(BASE, "TABELLA_LETTURE_BEAM.csv")

OUTPUT_DIR = "/content/drive/MyDrive/Contatori/OUTPUT/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for label, path in [("SAP", PATH_SAP), ("BEAM", PATH_BEAM), ("LETTURE", PATH_LETT)]:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {label}: {path}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 2 — Installazione librerie
# ══════════════════════════════════════════════════════════════════════════════

!pip install -q statsmodels pyarrow


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — Import e configurazione grafica
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

plt.rcParams.update({
    "figure.dpi": 120, "figure.figsize": (12, 5),
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "axes.titlesize": 13, "axes.labelsize": 11,
})
PALETTE = sns.color_palette("muted")
sns.set_style("whitegrid")
print("✅ Librerie importate correttamente.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 4 — Caricamento dati grezzi
# ══════════════════════════════════════════════════════════════════════════════

def load_csv(path, label):
    for enc in ["utf-8", "latin-1", "cp1252", "utf-8-sig"]:
        try:
            df = pd.read_csv(path, sep=";", encoding=enc, dtype=str, low_memory=False)
            df.columns = df.columns.str.strip()
            print(f"  ✅ {label}: {df.shape[0]:,} righe × {df.shape[1]} colonne  (encoding: {enc})")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Impossibile leggere {path} con nessun encoding provato.")

print("── Caricamento file ──")
df_sap  = load_csv(PATH_SAP,  "SAP")
df_beam = load_csv(PATH_BEAM, "BEAM")
df_lett = load_csv(PATH_LETT, "LETTURE")

df_lett = df_lett.drop(columns=["Unnamed: 6"], errors="ignore")

print("\n── Colonne SAP:", df_sap.columns.tolist())
print("── Colonne BEAM:", df_beam.columns.tolist())
print("── Colonne LETTURE:", df_lett.columns.tolist())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 5 — Ispezione
# ══════════════════════════════════════════════════════════════════════════════

for label, df in [("SAP", df_sap), ("BEAM", df_beam), ("LETTURE", df_lett)]:
    print("═" * 60)
    print(f"{label} — prime 3 righe")
    display(df.head(3))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 6 — Mapping nomi colonne
# ══════════════════════════════════════════════════════════════════════════════

COL_PROV         = "04IND - Provincia (cod)"
COL_COMUNE       = "04IND - Comune (cod)"
COL_CAP          = "04IND - CAP (cod)"
COL_ACCESSIBILE  = "03IMPS - STec - Accessibile (cod)"
COL_TELEGESTIONE = "03IMPS - Stato telegestione (cod)"
COL_CONSUMO      = "03IMPS - Operandi - Cons.Anno PDR (desc)"
COL_DATA_INST    = "11MDEF - Installazione(dta)"
COL_COSTRUTTORE  = "11MDEF - Costruttore (cod)"
COL_ANNO_COSTR   = "11MDEF - Anno Costruzione (cod)"
COL_MATERIALE    = "11MDEF - Materiale (cod)"
KEY_SAP          = "11MDEF - Numero Serie (cod)"
KEY_BEAM         = "MATRICOLA CONTATORE"
COL_MODELLO      = "MODELLO CONTATORE"
COL_FIRMWARE     = "VERSIONE FIRMWARE"
COL_TECN_COM     = "Tecnologia di comunicazione"
COL_ULT_COM      = "Data ultima comunicazione"
COL_ULT_MIS      = "Data ultima misura"
KEY_LETT         = "Matricola Contatore"
COL_DATA_LETT    = "Data lettura"
COL_STATO_LETT   = "Stato lettura"
COL_DIAGNOSTICA  = "Diagnostica Contatore"

print("Verifica colonne chiave:")
for df, col, label in [
    (df_sap,  KEY_SAP,       "SAP — matricola"),
    (df_beam, KEY_BEAM,      "BEAM — matricola"),
    (df_lett, KEY_LETT,      "LETTURE — matricola"),
    (df_lett, COL_DATA_LETT, "LETTURE — data lettura"),
    (df_sap,  COL_PROV,      "SAP — provincia"),
]:
    ok = col in df.columns
    print(f"  {'✅' if ok else '❌  ← DA CORREGGERE'} {label}: '{col}'")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 7 — Filtraggio Reggio Emilia
# ══════════════════════════════════════════════════════════════════════════════

print("\nValori unici colonna provincia (SAP):")
print(df_sap[COL_PROV].value_counts().to_string())

PROV_TARGET = "Reggio nell'Emilia"
df_sap_re = df_sap[df_sap[COL_PROV].str.contains(PROV_TARGET, case=False, na=False)].copy()

print(f"\nContatori SAP — provincia RE: {len(df_sap_re):,}")
if len(df_sap_re) == 0:
    print("⚠️  Nessun contatore trovato. Controlla PROV_TARGET.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 8 — Merge anagrafica e parco contatori RE
# ══════════════════════════════════════════════════════════════════════════════

df_sap_re["_key"] = df_sap_re[KEY_SAP].str.strip().str.upper()
df_beam["_key"]   = df_beam[KEY_BEAM].str.strip().str.upper()
df_lett["_key"]   = df_lett[KEY_LETT].str.strip().str.upper()

print(f"Contatori unici SAP-RE  : {df_sap_re['_key'].nunique():,}")
print(f"Contatori unici BEAM    : {df_beam['_key'].nunique():,}")
print(f"Contatori unici LETTURE : {df_lett['_key'].nunique():,}")

df_anagrafica = df_sap_re.merge(df_beam, on="_key", how="inner", suffixes=("_sap", "_beam"))

n_prima = len(df_anagrafica)
df_anagrafica = df_anagrafica.drop_duplicates(subset="_key", keep="first")
n_dedup = n_prima - len(df_anagrafica)
if n_dedup > 0:
    print(f"\n⚠️  Rimossi {n_dedup} duplicati di _key dopo il merge SAP↔BEAM")

print(f"\nDopo merge SAP↔BEAM (inner) : {len(df_anagrafica):,} contatori")
print(f"Scartati per anagrafica incompleta: {df_sap_re['_key'].nunique() - len(df_anagrafica):,}")

df_lett_re = df_lett[df_lett["_key"].isin(df_anagrafica["_key"])].copy()

matricole_parco      = set(df_anagrafica["_key"])
matricole_con_lett   = set(df_lett_re["_key"])
matricole_senza_lett = matricole_parco - matricole_con_lett
n_tot, n_con, n_senza = len(matricole_parco), len(matricole_con_lett), len(matricole_senza_lett)

print(f"\nParco contatori RE totale     : {n_tot:,}")
print(f"  ├─ Con almeno una lettura   : {n_con:,}  ({n_con/n_tot*100:.1f}%)")
print(f"  └─ Senza alcuna lettura     : {n_senza:,}  ({n_senza/n_tot*100:.1f}%)")
print(f"\nLetture filtrate (RE)         : {len(df_lett_re):,} righe")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 9 — Parsing date Oracle (vettorizzato)
# ══════════════════════════════════════════════════════════════════════════════

df_lett_re["data_lettura"] = pd.to_datetime(
    df_lett_re[COL_DATA_LETT].str.strip().str.split(" ").str[0],
    format="%d-%b-%y", errors="coerce"
)

n_nat = df_lett_re["data_lettura"].isna().sum()
print(f"Parsate correttamente : {df_lett_re['data_lettura'].notna().sum():,}")
print(f"Non parsabili         : {n_nat:,}")

df_lett_re = df_lett_re.dropna(subset=["data_lettura"])

DATA_MIN = df_lett_re["data_lettura"].min()
DATA_MAX = df_lett_re["data_lettura"].max()
DURATA   = (DATA_MAX - DATA_MIN).days

print(f"Prima lettura : {DATA_MIN.date()}")
print(f"Ultima lettura: {DATA_MAX.date()}")
print(f"Durata dataset: {DURATA} giorni ({DURATA/30:.1f} mesi)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 10 — Target a finestre di 3 giorni (vettorizzato)
# ══════════════════════════════════════════════════════════════════════════════

WINDOW_DAYS = 3

finestre = []
t = DATA_MIN
while t <= DATA_MAX:
    finestre.append((t, t + timedelta(days=WINDOW_DAYS - 1)))
    t += timedelta(days=WINDOW_DAYS)

print(f"Finestre costruite: {len(finestre)}")
for i, (ts, te) in enumerate(finestre):
    print(f"  W{i+1}: {ts.date()} → {te.date()}")

bins = [ts for ts, _ in finestre] + [finestre[-1][1] + timedelta(days=1)]
df_lett_re["finestra"] = pd.cut(
    df_lett_re["data_lettura"], bins=bins,
    labels=range(1, len(finestre) + 1), right=False
).astype("Int64")

comunicati = (
    df_lett_re.dropna(subset=["finestra"])
    .drop_duplicates(subset=["_key", "finestra"])[["_key", "finestra"]]
    .assign(silente=0)
)

meta = pd.DataFrame({
    "finestra": range(1, len(finestre) + 1),
    "t_start":  [ts for ts, _ in finestre],
    "t_end":    [te for _, te in finestre],
})

idx = pd.MultiIndex.from_product(
    [df_anagrafica["_key"].values, range(1, len(finestre) + 1)],
    names=["_key", "finestra"]
)
df_target = pd.DataFrame(index=idx).reset_index()
df_target = df_target.merge(comunicati, on=["_key", "finestra"], how="left")
df_target["silente"] = df_target["silente"].fillna(1).astype(int)
df_target = df_target.merge(meta, on="finestra", how="left")

print(f"\nShape dataset target: {df_target.shape}")
ct = df_target["silente"].value_counts()
print(f"  Comunicato (0): {ct.get(0,0):,}  ({ct.get(0,0)/len(df_target)*100:.1f}%)")
print(f"  Silente    (1): {ct.get(1,0):,}  ({ct.get(1,0)/len(df_target)*100:.1f}%)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 11 — Cross-section aggregata + salvataggio parquet
# ══════════════════════════════════════════════════════════════════════════════

# ── Cross-section: collassa panel → una riga per contatore ───────────────────
agg = (
    df_target.groupby("_key")["silente"]
    .agg(pct_silente="mean", n_finestre="count")
    .reset_index()
)
agg["silente_prevalente"] = (agg["pct_silente"] > 0.5).astype(int)
df_cs = df_anagrafica.merge(agg, on="_key", how="left")

print(f"df_cs shape: {df_cs.shape}")
print(f"  pct_silente media        : {df_cs['pct_silente'].mean():.3f}")
print(f"  silente_prevalente = 1   : {df_cs['silente_prevalente'].sum():,}  ({df_cs['silente_prevalente'].mean()*100:.1f}%)")

# ── Salvataggio parquet su Drive ──────────────────────────────────────────────
import os
PROC = OUTPUT_DIR

df_anagrafica.to_parquet(PROC + "df_anagrafica.parquet", index=False)
df_lett_re   .to_parquet(PROC + "df_lett_re.parquet",    index=False)
df_target    .to_parquet(PROC + "df_target.parquet",     index=False)
df_cs        .to_parquet(PROC + "df_cs.parquet",         index=False)

print("\n✅ Parquet salvati in:", PROC)
for f in ["df_anagrafica.parquet", "df_lett_re.parquet", "df_target.parquet", "df_cs.parquet"]:
    size = os.path.getsize(PROC + f) / 1024**2
    print(f"  {f}: {size:.1f} MB")
